In [1]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers peft tqdm

In [4]:
import torch
import json
from tqdm.auto import tqdm
from collections import Counter, defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [2]:
# ── 1. Config — SET THESE BEFORE RUNNING ─────────────────────────────────────
MODEL_ID     = "zou-lab/BioMed-R1-8B"

# FIX 1: ADAPTER_PATH was missing — this is where finetune script saved the adapter
ADAPTER_PATH = "/content/drive/MyDrive/BioMed AI CW/biomed_r1_finetuned/final_adapter"

# Choose which dataset to evaluate on:
#   dev_set.json  → 50 samples  → direct comparison with your 78% zero-shot result
#   test_set.json → 500 samples → main result to report
TEST_DATA_PATH = "/content/drive/MyDrive/BioMed AI CW/test_set.json"
# TEST_DATA_PATH = "/content/drive/MyDrive/BioMed AI CW/test_set.json"

RESULTS_FILE         = "finetuned_results.json"
MAX_REASONING_TOKENS = 768

In [5]:
# ── 2. Load Base Model + LoRA Adapter ────────────────────────────────────────
from transformers import BitsAndBytesConfig
print(f"Loading base model: {MODEL_ID}")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer from the adapter directory (it was saved there by the finetune script)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

# FIX 3: Load the LoRA adapter on top of the quantised base model
print(f"Loading LoRA adapter from: {ADAPTER_PATH}")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model ready.\n")

Loading base model: zou-lab/BioMed-R1-8B


config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading LoRA adapter from: /content/drive/MyDrive/BioMed AI CW/biomed_r1_finetuned/final_adapter
Model ready.



In [6]:
# ── 3. Load Test Data ─────────────────────────────────────────────────────────
with open(TEST_DATA_PATH, 'r') as f:
    test_data = json.load(f)

print(f"Evaluating on: {TEST_DATA_PATH.split('/')[-1]}  ({len(test_data)} samples)")

gold_labels = []
for entry in test_data.values():
    g = entry.get('final_decision', '')
    if isinstance(g, list): g = g[0]
    gold_labels.append(g.lower().strip())
print(f"Gold distribution: {Counter(gold_labels)}\n")

Evaluating on: test_set.json  (500 samples)
Gold distribution: Counter({'yes': 276, 'no': 169, 'maybe': 55})



In [7]:

# ── 4. Helpers (identical to zero-shot v3) ────────────────────────────────────
def build_context(labels: list, contexts: list) -> str:
    return "\n".join(
        f"{lbl.strip().title()}: {ctx.strip()}"
        for lbl, ctx in zip(labels, contexts)
    )

def get_prompt(question: str, context: str) -> str:
    return (
        f"The following is a structured biomedical abstract:\n\n"
        f"{context}\n\n"
        f"Based ONLY on the evidence in the abstract above, answer this question:\n"
        f"{question}\n\n"
        f"Instructions:\n"
        f"- Think through the evidence step by step.\n"
        f"- Your final answer MUST be exactly one of: yes, no, or maybe.\n"
        f"- End your response with exactly one of these three lines:\n"
        f"  Final Answer: yes\n"
        f"  Final Answer: no\n"
        f"  Final Answer: maybe\n"
    )

CONCLUSION_SIGNALS = {
    "yes": [
        "therefore yes", "so yes", "answer is yes", "conclude yes",
        "implies yes", "suggests yes", "indicates yes", "supports yes",
        "affirmative", "this is correct", "this is true", "this is supported",
        "effective", "beneficial", "successful", "accurate", "useful",
        "is justified", "is justifiable", "does help", "does work",
        "confirms", "demonstrates", "shows that", "proves",
        "implies affirmative", "the answer should be yes",
    ],
    "no": [
        "therefore no", "so no", "answer is no", "conclude no",
        "implies no", "suggests no", "indicates no", "not supported",
        "not effective", "not beneficial", "not accurate", "not useful",
        "is not justified", "does not help", "does not work",
        "fails to", "no evidence", "no significant", "not significant",
        "the answer should be no",
    ],
    "maybe": [
        "therefore maybe", "so maybe", "answer is maybe", "conclude maybe",
        "unclear", "uncertain", "inconclusive", "mixed evidence",
        "limited evidence", "conflicting", "insufficient evidence",
        "cannot be determined", "hard to say", "it depends",
        "the answer should be maybe",
    ],
}

def extract_answer(generated_text: str) -> tuple:
    text = generated_text.lower()

    # Stage 1: look for explicit "Final Answer:" marker (take first occurrence)
    if "final answer:" in text:
        answer_part = text.split("final answer:")[1][:30]
        if "yes"   in answer_part: return "yes",   "formal"
        if "no"    in answer_part: return "no",    "formal"
        if "maybe" in answer_part or "unsure" in answer_part:
            return "maybe", "formal"

    # Stage 2: scan tail of reasoning for conclusion signals
    tail = text[-400:]
    scores = {label: 0 for label in CONCLUSION_SIGNALS}
    for label, signals in CONCLUSION_SIGNALS.items():
        for signal in signals:
            if signal in tail:
                scores[label] += 1
    best = max(scores, key=scores.get)
    if scores[best] > 0:
        return best, "fallback"

    return "unknown", "unknown"

In [8]:

# ── 5. Inference Loop ─────────────────────────────────────────────────────────
results = []
print(f"Running inference...")

for entry in tqdm(test_data.values()):
    question     = entry.get('QUESTION', entry.get('question', ''))
    raw_labels   = entry.get('LABELS', [])
    raw_contexts = entry.get('CONTEXTS', [])

    if isinstance(raw_contexts, list) and isinstance(raw_labels, list):
        context = build_context(raw_labels, raw_contexts)
    elif isinstance(raw_contexts, list):
        context = " ".join(raw_contexts)
    else:
        context = str(raw_contexts)

    gold = entry.get('final_decision', '')
    if isinstance(gold, list): gold = gold[0]
    gold = gold.lower().strip()

    prompt       = get_prompt(question, context)
    inputs       = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_REASONING_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
        )

    generated_tokens = outputs[0][input_length:]
    generated_text   = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    prediction, method = extract_answer(generated_text)

    results.append({
        "generated_text": generated_text,
        "gold_label":     gold,
        "prediction":     prediction,
        "extract_method": method,
    })

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {RESULTS_FILE}\n")

Running inference...


  0%|          | 0/500 [00:00<?, ?it/s]

Results saved to finetuned_results.json



In [9]:

# ── 6. Metrics ────────────────────────────────────────────────────────────────
correct        = 0
pred_counts    = {"yes": 0, "no": 0, "maybe": 0, "unknown": 0}
gold_counts    = {"yes": 0, "no": 0, "maybe": 0}
correct_counts = {"yes": 0, "no": 0, "maybe": 0}
method_counts  = {"formal": 0, "fallback": 0, "unknown": 0}

for r in results:
    pred   = r["prediction"]
    actual = r["gold_label"]
    pred_counts[pred]  = pred_counts.get(pred, 0) + 1
    method_counts[r["extract_method"]] = method_counts.get(r["extract_method"], 0) + 1
    if actual in gold_counts:
        gold_counts[actual] += 1
    if pred == actual:
        correct += 1
        if actual in correct_counts:
            correct_counts[actual] += 1

total    = len(results)
accuracy = (correct / total * 100) if total > 0 else 0.0

# Macro F1
f1_scores = []
per_class  = {}
for cls in ["yes", "no", "maybe"]:
    tp = correct_counts[cls]
    fp = pred_counts[cls] - tp
    fn = gold_counts[cls]  - tp
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    f1_scores.append(f1)
    per_class[cls] = {"precision": p, "recall": r, "f1": f1}

macro_f1 = sum(f1_scores) / len(f1_scores)

print("─── Fine-Tuned Model Evaluation Results ──────────────────────")
print(f"  Model          : {MODEL_ID} + LoRA")
print(f"  Eval dataset   : {TEST_DATA_PATH.split('/')[-1]}")
print(f"  Accuracy       : {accuracy:.2f}%  ({correct}/{total})")
print(f"  Macro F1       : {macro_f1:.4f}")
print(f"\n  Gold distribution  : {gold_counts}")
print(f"  Prediction breakdown: {pred_counts}")
print(f"  Extraction method  : {method_counts}")
print("\n  Per-class metrics:")
for cls in ["yes", "no", "maybe"]:
    m = per_class[cls]
    print(f"    {cls:6s} — P: {m['precision']:.4f}  R: {m['recall']:.4f}  F1: {m['f1']:.4f}")

─── Fine-Tuned Model Evaluation Results ──────────────────────
  Model          : zou-lab/BioMed-R1-8B + LoRA
  Eval dataset   : test_set.json
  Accuracy       : 78.20%  (391/500)
  Macro F1       : 0.5483

  Gold distribution  : {'yes': 276, 'no': 169, 'maybe': 55}
  Prediction breakdown: {'yes': 284, 'no': 216, 'maybe': 0, 'unknown': 0}
  Extraction method  : {'formal': 500, 'fallback': 0, 'unknown': 0}

  Per-class metrics:
    yes    — P: 0.8380  R: 0.8623  F1: 0.8500
    no     — P: 0.7083  R: 0.9053  F1: 0.7948
    maybe  — P: 0.0000  R: 0.0000  F1: 0.0000


In [10]:

# ── 7. Confusion Matrix ───────────────────────────────────────────────────────
confusion = defaultdict(lambda: defaultdict(int))
for r in results:
    confusion[r["gold_label"]][r["prediction"]] += 1

print("\n  Confusion Matrix (rows=gold, cols=predicted):")
classes = ["yes", "no", "maybe", "unknown"]
print(f"  {'':8s}" + "".join(f"{c:10s}" for c in classes))
for gold_cls in ["yes", "no", "maybe"]:
    row = f"  {gold_cls:8s}"
    for pred_cls in classes:
        row += f"{confusion[gold_cls][pred_cls]:10d}"
    print(row)

print("\n─── Comparison Summary ────────────────────────────────────────")
print(f"  Zero-shot (dev set, 50 samples) : 78.00%")
print(f"  Fine-tuned ({TEST_DATA_PATH.split('/')[-1]:20s}) : {accuracy:.2f}%")
print("───────────────────────────────────────────────────────────────")


  Confusion Matrix (rows=gold, cols=predicted):
          yes       no        maybe     unknown   
  yes            238        38         0         0
  no              16       153         0         0
  maybe           30        25         0         0

─── Comparison Summary ────────────────────────────────────────
  Zero-shot (dev set, 50 samples) : 78.00%
  Fine-tuned (test_set.json       ) : 78.20%
───────────────────────────────────────────────────────────────
